# Homework 1: SwiftParcel Support Agent (hardened LangChain agent)

**Covers:** `create_agent`, Pydantic validation, trusted `ToolRuntime` context, business rules, timeouts and retries, idempotent writes, middleware, plus two new topics: **dynamic system prompts** and **structured output**.

**Time:** about 3 to 4 hours. **Submit:** this notebook with all cells run and the outputs visible.

## The story: SwiftParcel

**SwiftParcel** is a courier company. Customers write to support to ask where a parcel is, to move a delivery to another day, and to complain when a parcel is late, damaged or lost. The company wants an AI support agent, and you are the engineer building it.

You will build it twice, the same way we built the Zomato agent:

| Homework | What you build | Tools you use |
|---|---|---|
| **Homework 1** (this notebook) | A hardened LangChain agent that looks up parcels and reschedules deliveries | `create_agent`, Pydantic, `ToolRuntime`, middleware |
| **Homework 2** (next notebook) | A LangGraph version with memory, policy search and claim approval | LangGraph, PostgreSQL checkpoints, Qdrant, human approval |

### The data

Three customers (`301` Meera, GOLD; `302` Karan, STANDARD; `303` Divya, STANDARD) and ten parcels. Tracking ids look like `SP-10000001`. A parcel has a status (`PENDING_PICKUP`, `IN_TRANSIT`, `OUT_FOR_DELIVERY`, `DELIVERED`, `LOST`, `RETURNED`), a declared value, a promised date and a reschedule count.

### The business rules

| Rule | Value |
|---|---|
| A customer may see and change **only their own** parcels | always |
| Reschedule allowed when the status is | `PENDING_PICKUP` or `IN_TRANSIT` (**not** `OUT_FOR_DELIVERY`) |
| Reschedules per parcel | at most **2** |
| New delivery date | from tomorrow to **7 days** from today |
| Claims (Homework 2) | up to 500 rupees automatic, up to 5000 a reviewer, above 5000 a senior reviewer |

## How this homework works

- Cells marked **PROVIDED** work as they are. Do not change them.
- Cells marked **YOUR CODE** contain a stub. Replace the `TODO`.
- Under every task a **Must show** list says exactly which outputs to produce. A task without those outputs gets no marks, because "code that looks right" is not the goal. Demonstrated behaviour is.
- You may use anything from Notebook 2 as a starting point. Copy and adapt it; you are expected to.

| Task | Topic | Marks |
|---|---|---|
| 1 | Data layer and read tools | 15 |
| 2 | Reschedule tool: business rules and idempotency | 20 |
| 3 | The middleware stack | 20 |
| 4 | New topics: dynamic prompt and structured output | 15 |
| 5 | Failure drill and scenario table | 20 |
| | Code quality and the short write-ups | 10 |

## Setup (PROVIDED)


In [1]:
import os, sys, json, time, warnings
from datetime import date, timedelta

import psycopg
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

load_dotenv()
warnings.filterwarnings("ignore", message="Pydantic serializer warnings")
DATABASE_URL = os.environ["DATABASE_URL"]

llm = ChatOpenAI(
    model=os.environ["OPENROUTER_MODEL"],
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0, timeout=30, max_retries=2,
)

def ok(**data):
    return {"ok": True, **data}

def fail(code, message, **data):
    return {"ok": False, "code": code, "message": message, **data}

print("Ready. Model:", os.environ["OPENROUTER_MODEL"])

/Users/prathamjain/Desktop/Ai-Engg/Ai-Engg-RAG/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


Ready. Model: openai/gpt-4o-mini


In [3]:
# Create the SwiftParcel tables and (re)load the teaching data. Run this cell any time you want a clean start.
with psycopg.connect(DATABASE_URL) as conn:
    conn.execute(open("swiftparcel_schema.sql").read())
    conn.execute(open("swiftparcel_seed.sql").read())
    rows = conn.execute(
        "SELECT tracking_id, customer_id, status, declared_value, promised_date, reschedule_count "
        "FROM sp_parcels ORDER BY tracking_id"
    ).fetchall()
for row in rows:
    print(row)

('SP-10000001', 301, 'IN_TRANSIT', Decimal('2500.00'), datetime.date(2026, 10, 31), 0)
('SP-10000002', 301, 'OUT_FOR_DELIVERY', Decimal('1200.00'), datetime.date(2026, 10, 29), 0)
('SP-10000003', 301, 'DELIVERED', Decimal('800.00'), datetime.date(2026, 10, 27), 0)
('SP-10000004', 301, 'LOST', Decimal('3000.00'), datetime.date(2026, 10, 17), 0)
('SP-10000005', 302, 'IN_TRANSIT', Decimal('4500.00'), datetime.date(2026, 11, 1), 2)
('SP-10000006', 302, 'DELIVERED', Decimal('450.00'), datetime.date(2026, 10, 19), 0)
('SP-10000007', 302, 'PENDING_PICKUP', Decimal('6000.00'), datetime.date(2026, 11, 3), 0)
('SP-10000008', 303, 'IN_TRANSIT', Decimal('1200.00'), datetime.date(2026, 10, 30), 0)
('SP-10000009', 303, 'DELIVERED', Decimal('7500.00'), datetime.date(2026, 10, 28), 0)
('SP-10000010', 303, 'RETURNED', Decimal('900.00'), datetime.date(2026, 10, 23), 0)


In [4]:
# PROVIDED. A fake chat model that plays back scripted replies, so you can force failures on purpose.
# A step that is an Exception is raised instead of returned. The last step repeats when the script ends.
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatGeneration, ChatResult

class ScriptedChatModel(BaseChatModel):
    label: str = "scripted"
    script: list
    calls: int = 0

    @property
    def _llm_type(self) -> str:
        return "scripted"

    def bind_tools(self, tools, **kwargs):
        return self

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        step = self.script[min(self.calls, len(self.script) - 1)]
        self.calls += 1
        if isinstance(step, Exception):
            raise step
        return ChatResult(generations=[ChatGeneration(message=step)])

def tool_call(name, args, call_id="call-1"):
    return AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": call_id}])

## Task 1: Data layer and read tools (15 marks)

**Goal.** Give the agent two safe, read-only tools.

**Requirements**

1. `get_connection()` opens a connection with a **5 second connect timeout** and a **statement timeout** of 5 seconds.
2. `run_with_retry(fn, *args, attempts=3)` retries only `psycopg.OperationalError`, with a short growing pause, and re-raises after the last attempt. It must **not** retry any other error.
3. `query_parcel(customer_id, tracking_id)` and `query_recent_parcels(customer_id, limit)` return rows **only for that customer** (the ownership check is in the SQL `WHERE`, not in the prompt).
4. Pydantic schemas: `TrackingInput` (id must match `^SP-[0-9]{8}$`) and `RecentInput` (`limit` is an integer from 1 to 10, default 5).
5. Two tools, `get_parcel_status` and `list_my_parcels`. They read the customer from trusted context (`ToolRuntime`, with `context_schema=AgentContext` holding `customer_id` and `tier`). The model never supplies the customer id.
6. A parcel that does not exist and a parcel that belongs to someone else must return the **same** `PARCEL_NOT_FOUND` result, so the tool never confirms that another customer's parcel exists.
7. Database errors become a short `SERVICE_UNAVAILABLE` result. No SQL or host names in it.

**Must show** (run each through a small agent or call the query functions directly, and print the result)

| # | Call | Expected |
|---|---|---|
| a | customer 301 asks for `SP-10000001` | the parcel |
| b | customer 301 asks for `SP-10000005` (Karan's) | `PARCEL_NOT_FOUND` |
| c | customer 301 asks for `SP-99999999` | `PARCEL_NOT_FOUND` (identical to b) |
| d | tracking id `sp-1` | rejected by Pydantic before any SQL |
| e | `limit=0` and `limit=50` | both rejected |
| f | `list_my_parcels` for 301 | 4 parcels, newest first |

In [7]:
# YOUR CODE: Task 1
from typing import TypedDict
from psycopg.rows import dict_row
from pydantic import BaseModel, Field
from langchain.tools import ToolRuntime


class AgentContext(TypedDict):
    customer_id: int
    tier: str


def get_connection():
    return psycopg.connect(
        os.environ["DATABASE_URL"],
        connect_timeout=5,                      # 5s to establish the connection
        options="-c statement_timeout=5000",    # 5s per statement (milliseconds)
        row_factory=dict_row,
    )


def run_with_retry(fn, *args, attempts=3):
    for i in range(attempts):
        try:
            return fn(*args)
        except psycopg.OperationalError:        # only this error is retried
            if i == attempts - 1:
                raise
            time.sleep(0.2 * 2 ** i)


def query_parcel(customer_id, tracking_id):
    sql = """
        SELECT tracking_id, status, description, declared_value,
               promised_date, delivered_at, reschedule_count
        FROM sp_parcels
        WHERE tracking_id = %s AND customer_id = %s
    """
    with get_connection() as conn:
        return conn.execute(sql, (tracking_id, customer_id)).fetchone()


def query_recent_parcels(customer_id, limit):
    sql = """
        SELECT tracking_id, status, description, promised_date
        FROM sp_parcels
        WHERE customer_id = %s
        ORDER BY created_at DESC, tracking_id DESC
        LIMIT %s
    """
    with get_connection() as conn:
        return conn.execute(sql, (customer_id, limit)).fetchall()


def serialize(row):
    """Make a DB row JSON friendly (Decimal, date, datetime)."""
    return {
        k: (float(v) if k == "declared_value" else v.isoformat() if hasattr(v, "isoformat") else v)
        for k, v in row.items()
    }


class TrackingInput(BaseModel):
    tracking_id: str = Field(pattern=r"^SP-[0-9]{8}$",
                             description="Parcel tracking ID, e.g. SP-10000001")


class RecentInput(BaseModel):
    limit: int = Field(default=5, ge=1, le=10,
                       description="How many recent parcels to return (1-10)")


@tool(args_schema=TrackingInput)
def get_parcel_status(tracking_id: str, runtime: ToolRuntime[AgentContext]) -> dict:
    """Look up the status, promised date and reschedule count of one of the customer's own parcels."""
    customer_id = runtime.context["customer_id"]   # trusted context, never from the model
    try:
        row = run_with_retry(query_parcel, customer_id, tracking_id)
    except psycopg.Error:
        return fail("SERVICE_UNAVAILABLE", "Parcel service is temporarily unavailable. Please try again shortly.")
    if row is None:
        # Identical result for "does not exist" and "belongs to someone else": no information leak.
        return fail("PARCEL_NOT_FOUND", "No parcel with that tracking id was found on your account.")
    return ok(parcel=serialize(row))


@tool(args_schema=RecentInput)
def list_my_parcels(limit: int, runtime: ToolRuntime[AgentContext]) -> dict:
    """List the customer's most recent parcels (newest first)."""
    customer_id = runtime.context["customer_id"]
    try:
        rows = run_with_retry(query_recent_parcels, customer_id, limit)
    except psycopg.Error:
        return fail("SERVICE_UNAVAILABLE", "Parcel service is temporarily unavailable. Please try again shortly.")
    return ok(count=len(rows), parcels=[serialize(r) for r in rows])

In [8]:
# Task 1 demo: drive each tool through a small agent (scripted model, so no LLM cost) and print the tool result.
from langchain_core.messages import ToolMessage

def run_tool(customer_id, name, args, tier="STANDARD"):
    agent = create_agent(
        ScriptedChatModel(script=[tool_call(name, args), AIMessage(content="done")]),
        tools=[get_parcel_status, list_my_parcels],
        context_schema=AgentContext,
    )
    out = agent.invoke({"messages": [HumanMessage("hi")]},
                       context={"customer_id": customer_id, "tier": tier})
    return next(m for m in out["messages"] if isinstance(m, ToolMessage)).content

print("a) 301 -> SP-10000001:\n", run_tool(301, "get_parcel_status", {"tracking_id": "SP-10000001"}, "GOLD"))
print("\nb) 301 -> SP-10000005 (Karan's):\n", run_tool(301, "get_parcel_status", {"tracking_id": "SP-10000005"}, "GOLD"))
print("\nc) 301 -> SP-99999999:\n", run_tool(301, "get_parcel_status", {"tracking_id": "SP-99999999"}, "GOLD"))
print("\nd) tracking id 'sp-1':\n", run_tool(301, "get_parcel_status", {"tracking_id": "sp-1"}, "GOLD"))
print("\ne) limit=0:\n", run_tool(301, "list_my_parcels", {"limit": 0}, "GOLD"))
print("\ne) limit=50:\n", run_tool(301, "list_my_parcels", {"limit": 50}, "GOLD"))
print("\nf) list_my_parcels for 301:\n", run_tool(301, "list_my_parcels", {}, "GOLD"))


a) 301 -> SP-10000001:
 {"ok": true, "parcel": {"tracking_id": "SP-10000001", "status": "IN_TRANSIT", "description": "Running shoes", "declared_value": 2500.0, "promised_date": "2026-10-31", "delivered_at": null, "reschedule_count": 0}}

b) 301 -> SP-10000005 (Karan's):
 {"ok": false, "code": "PARCEL_NOT_FOUND", "message": "No parcel with that tracking id was found on your account."}

c) 301 -> SP-99999999:
 {"ok": false, "code": "PARCEL_NOT_FOUND", "message": "No parcel with that tracking id was found on your account."}

d) tracking id 'sp-1':
 Error invoking tool 'get_parcel_status' with kwargs {'tracking_id': 'sp-1'} with error:
 tracking_id: String should match pattern '^SP-[0-9]{8}$'
 Please fix the error and try again.

e) limit=0:
 Error invoking tool 'list_my_parcels' with kwargs {'limit': 0} with error:
 limit: Input should be greater than or equal to 1
 Please fix the error and try again.

e) limit=50:
 Error invoking tool 'list_my_parcels' with kwargs {'limit': 50} with erro

## Task 2: The reschedule tool (20 marks)

**Goal.** A write tool that is safe to call twice.

**Requirements**

1. `request_reschedule(tracking_id, new_date)` is a tool. Customer comes from trusted context. `new_date` is a `date`.
2. Validate in this order and return a clear `fail(...)` code for each:

   | Code | When |
   |---|---|
   | `PARCEL_NOT_FOUND` | not found, or not owned by this customer |
   | `NOT_RESCHEDULABLE` | status is not `PENDING_PICKUP` or `IN_TRANSIT` |
   | `LIMIT_REACHED` | the parcel already has 2 reschedules |
   | `DATE_OUT_OF_RANGE` | the date is not between tomorrow and 7 days from today |

3. **Idempotency.** Build a key from `customer_id`, `tracking_id` and `new_date` (for example a SHA-256 hash). Insert into `sp_reschedules` with `ON CONFLICT (idempotency_key) DO NOTHING`. Only when a row was really inserted, in the **same transaction**, increase `reschedule_count` and set `promised_date`. Repeating the same request returns `ok(code="ALREADY_REQUESTED")` and changes nothing.
4. Concurrency: two simultaneous identical requests must still produce one row and one count increase. In 3 sentences, explain why a unique key plus one transaction achieves that, and why "check first, then insert" does not.

**Must show.** Reset the data with the seed cell, then run these and print the result and the final database rows:

| # | Request | Expected |
|---|---|---|
| a | customer 301 reschedules `SP-10000001` to tomorrow + 1 day | `RESCHEDULED`, count 1 |
| b | the same request again | `ALREADY_REQUESTED`, count still 1, one row |
| c | a different valid date | `RESCHEDULED`, count 2 |
| d | a third valid date | `LIMIT_REACHED` |
| e | 301 reschedules `SP-10000002` (out for delivery) | `NOT_RESCHEDULABLE` |
| f | customer 303 reschedules `SP-10000008` to a date 20 days away | `DATE_OUT_OF_RANGE` |
| g | customer 301 reschedules Karan's `SP-10000007` | `PARCEL_NOT_FOUND` |

In [9]:
# YOUR CODE: Task 2
import hashlib

RESCHEDULABLE = ("PENDING_PICKUP", "IN_TRANSIT")
MAX_RESCHEDULES = 2
MAX_DAYS_AHEAD = 7


def idempotency_key(customer_id, tracking_id, new_date):
    raw = f"{customer_id}|{tracking_id}|{new_date.isoformat()}"
    return hashlib.sha256(raw.encode()).hexdigest()


def _create_reschedule(customer_id, tracking_id, new_date):
    key = idempotency_key(customer_id, tracking_id, new_date)
    with get_connection() as conn:                 # one transaction: commits on success, rolls back on error
        # FOR UPDATE locks the parcel row, so concurrent requests for it queue up behind each other.
        parcel = conn.execute(
            "SELECT status, reschedule_count FROM sp_parcels "
            "WHERE tracking_id = %s AND customer_id = %s FOR UPDATE",
            (tracking_id, customer_id),
        ).fetchone()
        if parcel is None:
            return fail("PARCEL_NOT_FOUND", "No parcel with that tracking id was found on your account.")

        # A replay of a request that already succeeded must stay a no-op, even if the parcel
        # has since reached the limit or the date has drifted out of range.
        seen = conn.execute("SELECT 1 FROM sp_reschedules WHERE idempotency_key = %s", (key,)).fetchone()
        if seen:
            return ok(code="ALREADY_REQUESTED", tracking_id=tracking_id, new_date=new_date.isoformat(),
                      reschedule_count=parcel["reschedule_count"],
                      message="This reschedule was already requested. Nothing was changed.")

        if parcel["status"] not in RESCHEDULABLE:
            return fail("NOT_RESCHEDULABLE",
                        f"A parcel with status {parcel['status']} cannot be rescheduled.")
        if parcel["reschedule_count"] >= MAX_RESCHEDULES:
            return fail("LIMIT_REACHED", f"This parcel has already been rescheduled {MAX_RESCHEDULES} times.")
        today = date.today()
        if not (today + timedelta(days=1) <= new_date <= today + timedelta(days=MAX_DAYS_AHEAD)):
            return fail("DATE_OUT_OF_RANGE",
                        f"The new date must be between {today + timedelta(days=1)} and "
                        f"{today + timedelta(days=MAX_DAYS_AHEAD)}.")

        inserted = conn.execute(
            "INSERT INTO sp_reschedules (tracking_id, customer_id, new_date, idempotency_key) "
            "VALUES (%s, %s, %s, %s) ON CONFLICT (idempotency_key) DO NOTHING",
            (tracking_id, customer_id, new_date, key),
        ).rowcount
        if inserted == 0:                          # lost a race to an identical request
            return ok(code="ALREADY_REQUESTED", tracking_id=tracking_id, new_date=new_date.isoformat(),
                      reschedule_count=parcel["reschedule_count"],
                      message="This reschedule was already requested. Nothing was changed.")

        updated = conn.execute(
            "UPDATE sp_parcels SET reschedule_count = reschedule_count + 1, promised_date = %s "
            "WHERE tracking_id = %s AND customer_id = %s RETURNING reschedule_count",
            (new_date, tracking_id, customer_id),
        ).fetchone()
        return ok(code="RESCHEDULED", tracking_id=tracking_id, new_date=new_date.isoformat(),
                  reschedule_count=updated["reschedule_count"])


def create_reschedule(customer_id, tracking_id, new_date):
    """Validate and apply a reschedule. Safe to retry: the idempotency key makes repeats no-ops."""
    try:
        return run_with_retry(_create_reschedule, customer_id, tracking_id, new_date)
    except psycopg.Error:
        return fail("SERVICE_UNAVAILABLE", "Reschedule service is temporarily unavailable. Please try again shortly.")


class RescheduleInput(BaseModel):
    tracking_id: str = Field(pattern=r"^SP-[0-9]{8}$",
                             description="Parcel tracking ID, e.g. SP-10000001")
    new_date: date = Field(description="New delivery date, YYYY-MM-DD, from tomorrow to 7 days from today")


@tool(args_schema=RescheduleInput)
def request_reschedule(tracking_id: str, new_date: date, runtime: ToolRuntime[AgentContext]) -> dict:
    """Move the delivery of one of the customer's own parcels to a new date."""
    return create_reschedule(runtime.context["customer_id"], tracking_id, new_date)


**Why a unique key plus one transaction is safe under concurrency.** The `UNIQUE` constraint on `idempotency_key` is enforced by the database itself, so when two identical requests race, exactly one `INSERT` succeeds and the other hits `ON CONFLICT DO NOTHING` with `rowcount = 0`. Because the count update runs in the same transaction and only when a row was really inserted, the row and the `reschedule_count` increase commit or roll back together, giving one row and one increase. "Check first, then insert" fails because both requests can run the check before either has inserted, both see "not yet requested", and both then insert and increase the count (a time-of-check to time-of-use race).


In [10]:
# Task 2 demo: reset the data, run cases a-g, then print the final database rows.
with psycopg.connect(DATABASE_URL) as conn:
    conn.execute(open("swiftparcel_schema.sql").read())
    conn.execute(open("swiftparcel_seed.sql").read())

tomorrow = date.today() + timedelta(days=1)
cases = [
    ("a) 301 SP-10000001 -> tomorrow+1",        301, "SP-10000001", tomorrow + timedelta(days=1)),
    ("b) same request again",                   301, "SP-10000001", tomorrow + timedelta(days=1)),
    ("c) a different valid date",               301, "SP-10000001", tomorrow + timedelta(days=2)),
    ("d) a third valid date",                   301, "SP-10000001", tomorrow + timedelta(days=3)),
    ("e) 301 SP-10000002 (out for delivery)",   301, "SP-10000002", tomorrow + timedelta(days=1)),
    ("f) 303 SP-10000008 -> 20 days away",      303, "SP-10000008", date.today() + timedelta(days=20)),
    ("g) 301 reschedules Karan's SP-10000007",  301, "SP-10000007", tomorrow + timedelta(days=1)),
]
for label, cust, tid, d in cases:
    print(label, "->", create_reschedule(cust, tid, d))

with get_connection() as conn:
    print("\nsp_reschedules:")
    for r in conn.execute("SELECT tracking_id, customer_id, new_date, left(idempotency_key, 12) AS key12 "
                          "FROM sp_reschedules ORDER BY id"):
        print(" ", r)
    print("sp_parcels (touched):")
    for r in conn.execute("SELECT tracking_id, status, promised_date, reschedule_count FROM sp_parcels "
                          "WHERE tracking_id IN ('SP-10000001','SP-10000002','SP-10000007','SP-10000008') "
                          "ORDER BY tracking_id"):
        print(" ", r)

# Concurrency check: 8 identical simultaneous requests for Karan's SP-10000007 -> one row, one increase.
from concurrent.futures import ThreadPoolExecutor
same_date = tomorrow + timedelta(days=4)
with ThreadPoolExecutor(8) as pool:
    results = list(pool.map(lambda _: create_reschedule(302, "SP-10000007", same_date), range(8)))
print("\nconcurrent codes:", sorted(r["code"] for r in results))
with get_connection() as conn:
    print(conn.execute("SELECT count(*) AS rows FROM sp_reschedules WHERE tracking_id = 'SP-10000007'").fetchone(),
          conn.execute("SELECT reschedule_count FROM sp_parcels WHERE tracking_id = 'SP-10000007'").fetchone())


a) 301 SP-10000001 -> tomorrow+1 -> {'ok': True, 'code': 'RESCHEDULED', 'tracking_id': 'SP-10000001', 'new_date': '2026-11-01', 'reschedule_count': 1}
b) same request again -> {'ok': True, 'code': 'ALREADY_REQUESTED', 'tracking_id': 'SP-10000001', 'new_date': '2026-11-01', 'reschedule_count': 1, 'message': 'This reschedule was already requested. Nothing was changed.'}
c) a different valid date -> {'ok': True, 'code': 'RESCHEDULED', 'tracking_id': 'SP-10000001', 'new_date': '2026-11-02', 'reschedule_count': 2}
d) a third valid date -> {'ok': False, 'code': 'LIMIT_REACHED', 'message': 'This parcel has already been rescheduled 2 times.'}
e) 301 SP-10000002 (out for delivery) -> {'ok': False, 'code': 'NOT_RESCHEDULABLE', 'message': 'A parcel with status OUT_FOR_DELIVERY cannot be rescheduled.'}
f) 303 SP-10000008 -> 20 days away -> {'ok': False, 'code': 'DATE_OUT_OF_RANGE', 'message': 'The new date must be between 2026-10-31 and 2026-11-06.'}
g) 301 reschedules Karan's SP-10000007 -> {'ok'

## Task 3: The middleware stack (20 marks)

**Goal.** Put every cross-cutting rule in middleware instead of inside the tools.

**Requirements.** Build `parcel_agent` with `create_agent(...)` and a `middleware=[...]` list containing all of these:

| Middleware | Setting |
|---|---|
| your own `audit_tool_calls` (`@wrap_tool_call`) | appends `{tool, args, status, ms}` to a list `audit_log` for every tool call |
| your own `safe_tool_errors` (`@wrap_tool_call`) | catches errors, logs the detail, returns a short safe message to the model |
| `ModelCallLimitMiddleware` | `run_limit=6`, end gracefully |
| `ToolCallLimitMiddleware` for `request_reschedule` | `run_limit=1`: the agent cannot reschedule twice in one run |
| `ModelFallbackMiddleware` and `ModelRetryMiddleware` | retry the main model twice, then switch to a backup model |
| `ToolRetryMiddleware` | **read tools only**, transient database errors only, `on_failure="error"` |
| `PIIMiddleware` for email | `redact` |
| `PIIMiddleware` for Indian mobile numbers | `mask`, with your own regex (for example `(?:\+91[\s-]?)?[6-9]\d{4}[\s-]?\d{5}`) |
| `SummarizationMiddleware` | trigger after 6 messages, keep 2 (small on purpose, so you can see it fire) |

**Write-up (at most 5 sentences).** State your order and why: which middleware must come before which, and where `audit_tool_calls` sits relative to the retry layer (does it see one call or every attempt, and which do you want?).

**Must show.** Use the provided `ScriptedChatModel` and temporary patches so each policy fires **on purpose**:

| # | Demonstrate | Evidence to print |
|---|---|---|
| a | **Tool retry** recovers: make `query_parcel` raise `psycopg.OperationalError` twice, then work | the final answer, and how many attempts happened |
| b | **Model fallback**: primary model raises `TimeoutError` every time | the backup model answers; primary call count is 3 |
| c | **PII**: send "my email is meera.nair@example.com and my phone is 98765 43210, where is SP-10000001?" | the text the model actually received (redacted email, masked phone) |
| d | **Tool-call limit**: a scripted model asks for `request_reschedule` twice in one run | only one executes; the second is blocked |
| e | **Audit log** | `audit_log` printed as a small table after a run with two tool calls |
| f | **Summarization**: run a 10-message history through the agent | message count before and after |

In [11]:
# YOUR CODE: Task 3 (middleware, then the agent)
import logging
import openai
from langchain.agents.middleware import (
    ModelCallLimitMiddleware, ToolCallLimitMiddleware, ModelFallbackMiddleware, ModelRetryMiddleware,
    ToolRetryMiddleware, PIIMiddleware, SummarizationMiddleware, wrap_tool_call,
)
from langchain_core.messages import ToolMessage

logger = logging.getLogger("swiftparcel")
audit_log = []


@wrap_tool_call
def audit_tool_calls(request, handler):
    """Record one {tool, args, status, ms} row per logical tool call (retries happen inside this)."""
    start = time.perf_counter()
    status = "ok"
    try:
        result = handler(request)
        if isinstance(result, ToolMessage) and result.status == "error":
            status = "error"
        return result
    except Exception:
        status = "error"
        raise
    finally:
        audit_log.append({
            "tool": request.tool_call["name"],
            "args": request.tool_call["args"],
            "status": status,
            "ms": round((time.perf_counter() - start) * 1000, 1),
        })


@wrap_tool_call
def safe_tool_errors(request, handler):
    """Log the real error, give the model only a short safe message (no SQL, hosts or stack traces)."""
    try:
        return handler(request)
    except Exception:
        logger.exception("tool %s failed", request.tool_call["name"])
        return ToolMessage(
            content=json.dumps(fail("TOOL_ERROR", "The tool failed. Please try again later.")),
            tool_call_id=request.tool_call["id"],
            name=request.tool_call["name"],
            status="error",
        )


READ_TOOLS = [get_parcel_status, list_my_parcels]
ALL_TOOLS = READ_TOOLS + [request_reschedule]
INDIAN_MOBILE = r"(?:\+91[\s-]?)?[6-9]\d{4}[\s-]?\d{5}"

backup_llm = llm   # no second model id available; same model stands in as the backup


def build_agent(model, backup, summarizer):
    return create_agent(
        model=model,
        tools=ALL_TOOLS,
        context_schema=AgentContext,
        middleware=[
            # before_model hooks run top to bottom: scrub PII first so nothing later (summaries, the model) sees it
            PIIMiddleware("email", strategy="redact"),
            PIIMiddleware("indian_mobile", detector=INDIAN_MOBILE, strategy="mask"),
            SummarizationMiddleware(model=summarizer, trigger=("messages", 6), keep=("messages", 2)),
            ModelCallLimitMiddleware(run_limit=6, exit_behavior="end"),
            ToolCallLimitMiddleware(tool_name="request_reschedule", run_limit=1),
            # tool wrappers nest in this order: audit > safe errors > retry > tool
            audit_tool_calls,
            safe_tool_errors,
            ToolRetryMiddleware(max_retries=2, tools=READ_TOOLS, retry_on=(psycopg.OperationalError,),
                                on_failure="error", initial_delay=0.1, jitter=False),
            # model wrappers nest: fallback > retry > model (retry twice on the main model, then switch)
            ModelFallbackMiddleware(backup),
            ModelRetryMiddleware(max_retries=2, on_failure="error", initial_delay=0.1, jitter=False,
                                 retry_on=(TimeoutError, ConnectionError, openai.APIConnectionError,
                                           openai.RateLimitError, openai.InternalServerError)),
        ],
    )


parcel_agent = build_agent(llm, backup_llm, llm)
print("parcel_agent ready with", len(parcel_agent.nodes), "graph nodes")


parcel_agent ready with 11 graph nodes


In [13]:
# YOUR CODE: Task 3 demonstrations (a to f)
import logging
logging.getLogger("swiftparcel").setLevel(logging.CRITICAL)   # keep the demo output short

class RecordingModel(ScriptedChatModel):
    """ScriptedChatModel that also remembers every message list it was called with."""
    seen: list = []

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        self.seen.append(list(messages))
        return super()._generate(messages, stop, run_manager, **kwargs)

def final(text):
    return AIMessage(content=text)

def scripted_agent(script, backup=None, summarizer=None):
    model = RecordingModel(script=script)
    agent = build_agent(model, backup or RecordingModel(label="backup", script=[final("backup answer")]),
                        summarizer or RecordingModel(label="summarizer", script=[final("SUMMARY")]))
    return agent, model

ctx301 = {"customer_id": 301, "tier": "GOLD"}
with psycopg.connect(DATABASE_URL) as conn:       # clean data for the demos
    conn.execute(open("swiftparcel_schema.sql").read())
    conn.execute(open("swiftparcel_seed.sql").read())

# a) tool retry recovers: query_parcel fails twice with OperationalError, then works
attempts = {"n": 0}
real_query_parcel = query_parcel
def flaky_query_parcel(customer_id, tracking_id):
    attempts["n"] += 1
    if attempts["n"] <= 2:
        raise psycopg.OperationalError("simulated transient failure")
    return real_query_parcel(customer_id, tracking_id)

query_parcel = flaky_query_parcel
try:
    agent, _ = scripted_agent([tool_call("get_parcel_status", {"tracking_id": "SP-10000001"}),
                               final("Your parcel SP-10000001 is IN_TRANSIT.")])
    out = agent.invoke({"messages": [HumanMessage("where is SP-10000001?")]}, context=ctx301)
finally:
    query_parcel = real_query_parcel
print("a) final answer:", out["messages"][-1].content)
print("   query_parcel attempts:", attempts["n"], "(2 failures + 1 success)")
print("   tool result:", [m.content for m in out["messages"] if isinstance(m, ToolMessage)][0][:80], "...")

# b) model fallback: primary raises TimeoutError every time
agent, primary = scripted_agent([TimeoutError("primary timed out")])
out = agent.invoke({"messages": [HumanMessage("hello")]}, context=ctx301)
print("\nb) answer:", out["messages"][-1].content, "| primary calls:", primary.calls)

# c) PII: print what the model actually received
agent, model = scripted_agent([final("ok")])
agent.invoke({"messages": [HumanMessage(
    "my email is meera.nair@example.com and my phone is 98765 43210, where is SP-10000001?")]}, context=ctx301)
print("\nc) model received:", model.seen[0][-1].content)

# d) tool-call limit: two request_reschedule calls in one run -> only the first executes
t = date.today()
agent, _ = scripted_agent([
    tool_call("request_reschedule", {"tracking_id": "SP-10000008", "new_date": str(t + timedelta(days=2))}, "c1"),
    tool_call("request_reschedule", {"tracking_id": "SP-10000008", "new_date": str(t + timedelta(days=3))}, "c2"),
    final("done"),
])
out = agent.invoke({"messages": [HumanMessage("move it twice")]}, context={"customer_id": 303, "tier": "STANDARD"})
print("\nd) tool messages:")
for m in out["messages"]:
    if isinstance(m, ToolMessage):
        print("  ", m.tool_call_id, m.status, "|", m.content[:110])
with get_connection() as conn:
    print("   rows in sp_reschedules for SP-10000008:",
          conn.execute("SELECT count(*) AS n FROM sp_reschedules WHERE tracking_id='SP-10000008'").fetchone()["n"])

# e) audit log after a run with two tool calls
audit_log.clear()
agent, _ = scripted_agent([tool_call("get_parcel_status", {"tracking_id": "SP-10000001"}, "c1"),
                           tool_call("list_my_parcels", {"limit": 3}, "c2"), final("done")])
agent.invoke({"messages": [HumanMessage("status then list")]}, context=ctx301)
print("\ne) audit_log")
print("   {:<20} {:<32} {:<7} {}".format("tool", "args", "status", "ms"))
for r in audit_log:
    print("   {:<20} {:<32} {:<7} {}".format(r["tool"], json.dumps(r["args"]), r["status"], r["ms"]))

# f) summarization: 10-message history
history = []
for i in range(5):
    history += [HumanMessage(f"question {i}"), AIMessage(content=f"answer {i}")]
agent, model = scripted_agent([final("ok")])
out = agent.invoke({"messages": history}, context=ctx301)
print("\nf) messages before:", len(history), "| messages the model received:", len(model.seen[0]),
      "| first one:", str(model.seen[0][0].content)[:600])


a) final answer: Your parcel SP-10000001 is IN_TRANSIT.
   query_parcel attempts: 3 (2 failures + 1 success)
   tool result: {"ok": true, "parcel": {"tracking_id": "SP-10000001", "status": "IN_TRANSIT", "d ...

b) answer: backup answer | primary calls: 3

c) model received: my email is [REDACTED_EMAIL] and my phone is ****3210, where is SP-10000001?

d) tool messages:
   c1 success | {"ok": true, "code": "RESCHEDULED", "tracking_id": "SP-10000008", "new_date": "2026-11-02", "reschedule_count"
   c2 error | Tool call limit exceeded. Do not call 'request_reschedule' again.
   rows in sp_reschedules for SP-10000008: 1

e) audit_log
   tool                 args                             status  ms
   get_parcel_status    {"tracking_id": "SP-10000001"}   ok      6.3
   list_my_parcels      {"limit": 3}                     ok      6.6

f) messages before: 10 | messages the model received: 3 | first one: Here is a summary of the conversation to date:

SUMMARY


**Write-up (at most 5 sentences): your middleware order and why.**

The two PII middlewares come first so that nothing later, including the summaries and the model itself, ever sees a raw email or phone number, and summarization comes before the call limits so they count the real model calls. For tools I nest `audit_tool_calls` outside `safe_tool_errors`, which is outside `ToolRetryMiddleware`: retries run first, and only when they are exhausted does `safe_tool_errors` turn the exception into a short safe message, which is why the retry layer uses `on_failure="error"` so that it raises instead of hiding the failure. Because audit sits outside the retry layer it records one row per logical call (with the total time including retries); I want that for an audit trail, whereas putting it inside would log every attempt, which is noisier and better suited to debugging. For models, `ModelFallbackMiddleware` is outside `ModelRetryMiddleware`, so the primary model is retried twice and only then is the backup tried. `ToolRetryMiddleware` is limited to the read tools, because retrying `request_reschedule` blindly is only safe thanks to its idempotency key and a write should never be retried by default.


## Task 4: New topics, a dynamic prompt and structured output (15 marks)

### 4a. Dynamic system prompt

So far the system prompt was a fixed string. The `@dynamic_prompt` decorator turns a function into middleware that **builds the prompt for each model call** from the request, including the trusted context.

```python
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dynamic_prompt
def tier_prompt(request: ModelRequest) -> str:
    tier = request.runtime.context["tier"]
    ...
```

Write `tier_prompt` so that:

- every customer gets the base rules (use tools, never invent parcel facts, a reschedule is real but a claim is never promised);
- **GOLD** customers are greeted by name (add `name` to the context) and told they may ask for a priority callback;
- **STANDARD** customers get a short, plain answer style.

### 4b. Structured output

Free text is hard for other software to use. Ask the agent for a typed final answer with `response_format`:

```python
from typing import Literal
from pydantic import BaseModel

class SupportReply(BaseModel):
    answer: str
    tracking_id: str | None
    action: Literal["NONE", "LOOKUP", "RESCHEDULED", "REFUSED"]
    needs_human: bool
```

Pass `response_format=SupportReply` to `create_agent`. The typed result appears in `result["structured_response"]`.

**Must show**

| # | Show |
|---|---|
| a | The system prompt text actually produced for a GOLD context and for a STANDARD context (call your function with a small fake request object, or log it). They must differ. |
| b | Three questions run as customer 301 (GOLD) and 302 (STANDARD): a lookup, a successful reschedule, a refused reschedule. For each, print `result["structured_response"]`. |
| c | One sentence: what in your pipeline makes `action` trustworthy, and what does **not**? (Hint: who decided the value, the model or your code?) |

In [ ]:
# YOUR CODE: Task 4
from typing import Literal
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class SupportReply(BaseModel):
    # TODO
    pass


@dynamic_prompt
def tier_prompt(request: ModelRequest) -> str:
    # TODO: use request.runtime.context["tier"] and ["name"]
    return "TODO"


# TODO: tiered_agent = create_agent(..., middleware=[tier_prompt, ...], response_format=SupportReply)

In [ ]:
# YOUR CODE: Task 4 demonstrations (a to c)